<a href="https://colab.research.google.com/github/ycui4407/cs250-fall2026-Yi-Cui/blob/main/Labs/L13_CUI_YI_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS250 · Lab 13 · Arguments & Scope

In this lab you will practice everything from this week's lecture:

| Part | Topics |
|---|---|
| **A · Arguments** (50 pts) | positional, keyword, keyword-only (`*`), defaults, the mutable-default trap, `*args`, `**kwargs`, `**` unpacking, parameter order |
| **B · Scope** (41 pts) | local, global & `global`, enclosed & `nonlocal`, built-in names, mutating vs reassigning, the **LEGB** rule |
| **C · Organizing programs** (9 pts) | refactoring copy-pasted code into functions |

### How this lab works
1. Run the **setup cell** below (▶️). If Colab restarts or disconnects, run it again.
2. Fill in your name in the **Student Info** cell and run it.
3. For each exercise, write your code in the code cell, run it, then run the **✔️ check cell** right after it.
   * ✅ means that test passed. ❌ tells you exactly which input failed and what was expected.
   * You can fix and re-check as many times as you like.
4. At the end, run the **🏁 Final Check** cell to see your grade. When you reach **100/100**, it will tell you how to submit.

> 💡 For the *predict* questions, decide on your answer **before** running anything. That is how you build the skill the exam tests.

In [ ]:
#@title ▶️ **Run this cell first** — loads the auto-grader (do not edit)
_L13_SRC = r'''
import hashlib, dis, inspect, io, contextlib, builtins

NB = {}          # notebook namespace (set by the setup cell)
RESULTS = {}     # ex number -> (earned, points)
ANS = {'q1_error': '4f41e38e0b184032', 'q8a': 'c6d092e9b700ac06', 'q8b': '776b935d323cdc0c', 'q8c': 'c35e32ee44c50e98', 'q8d': 'ca0f1ec935ee9186', 'q13': '6485e6b329d98bb7', 'q14': 'def13825662ee78a', 'q15a': '48469c1fac8d2bbf', 'q15b': '6b668752f91012b3', 'q15c': 'd364fa3680601d54', 'q15d': '7587cd79747a5165'}

P = inspect.Parameter


def _h(qid, ans):
    return hashlib.sha256(("CS250-L13|%s|%s" % (qid, str(ans).strip().upper())).encode()).hexdigest()[:16]


class _Missing(Exception):
    pass


def _get(name):
    if name not in NB:
        raise _Missing(name)
    return NB[name]


def _call(fn, *a, **k):
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            return ("ok", fn(*a, **k))
    except Exception as e:
        return ("err", e)


def _desc(res):
    if res[0] == "err":
        return "raised %s: %s" % (type(res[1]).__name__, res[1])
    return "returned %r" % (res[1],)


def _ops(fn):
    try:
        return [i.opname for i in dis.get_instructions(fn)]
    except Exception:
        return []


def _uses_global(fn):
    return "STORE_GLOBAL" in _ops(fn)


def _sig(fn):
    try:
        return inspect.signature(fn).parameters
    except Exception:
        return {}


class _C:
    def __init__(self):
        self.n = 0
        self.ok_n = 0
        self.lines = []

    def t(self, cond, good, bad):
        self.n += 1
        if cond:
            self.ok_n += 1
            self.lines.append("   ✅ " + good)
        else:
            self.lines.append("   ❌ " + bad)
        return cond

    def expect(self, label, res, expected):
        ok = res[0] == "ok" and res[1] == expected
        return self.t(ok, "%s → %r" % (label, expected),
                      "%s %s, but it should return %r" % (label, _desc(res), expected))

    def raises(self, label, res, exc):
        ok = res[0] == "err" and isinstance(res[1], exc)
        return self.t(ok, "%s raises %s (as it should)" % (label, exc.__name__),
                      "%s %s, but it should raise %s" % (label, _desc(res), exc.__name__))

    def mcq(self, qid, hint):
        val = NB.get(qid, "")
        if val is None or str(val).strip() == "":
            return self.t(False, "", "%s has not been answered yet" % qid)
        ok = _h(qid, val) == ANS[qid]
        return self.t(ok, "%s is correct" % qid, "%s = %r is not correct. Hint: %s" % (qid, val, hint))


# ---------------------------------------------------------------- PART A
def ex1(c):
    f = _get("trip_summary")
    c.expect('trip_summary("Lima", 5, 3)', _call(f, "Lima", 5, 3), "3 traveler(s), 5 night(s) in Lima")
    c.expect('trip_summary("Oslo", 1, 1)', _call(f, "Oslo", 1, 1), "1 traveler(s), 1 night(s) in Oslo")
    names = list(_sig(f))
    c.t(names == ["city", "nights", "travelers"], "parameters are (city, nights, travelers)",
        "parameters must be named (city, nights, travelers) in that order; found %s" % names)
    fixed = _get("fixed_trip")
    c.t(fixed == "2 traveler(s), 4 night(s) in Tokyo", "fixed_trip has the arguments in the right order",
        "fixed_trip is %r. Put the arguments in the order city, nights, travelers" % (fixed,))
    c.mcq("q1_error", "call trip_summary(\"Rome\", 3) yourself and read the first word of the error")


def ex2(c):
    f = _get("create_account")
    want = {"username": "ana", "email": "ana@sfbu.edu", "admin": False}
    c.expect('create_account("ana", email="ana@sfbu.edu")', _call(f, "ana", email="ana@sfbu.edu"), want)
    c.expect('create_account(email="ana@sfbu.edu", username="ana")',
             _call(f, email="ana@sfbu.edu", username="ana"), want)
    c.expect('create_account("li", email="li@sfbu.edu", admin=True)',
             _call(f, "li", email="li@sfbu.edu", admin=True),
             {"username": "li", "email": "li@sfbu.edu", "admin": True})
    c.raises('create_account("ana", "ana@sfbu.edu")', _call(f, "ana", "ana@sfbu.edu"), TypeError)
    ps = _sig(f)
    c.t("email" in ps and ps["email"].kind == P.KEYWORD_ONLY and ps["email"].default is P.empty,
        "email is a required keyword-only parameter",
        "email must come after a bare * so it is keyword-only, and it must have no default")
    acct = _get("my_account")
    c.t(isinstance(acct, dict) and acct.get("admin") is True and acct.get("email") and acct.get("username"),
        "my_account was created with admin=True",
        "my_account should be the dict returned by create_account(...) with admin=True")


def ex3(c):
    f = _get("ticket_price")
    c.expect("ticket_price(30)", _call(f, 30), 12)
    c.expect('ticket_price(30, "weekend")', _call(f, 30, "weekend"), 15)
    c.expect("ticket_price(8)", _call(f, 8), 6)
    c.expect('ticket_price(70, day="weekend")', _call(f, 70, day="weekend"), 7.5)
    c.expect("ticket_price(20, student=True)", _call(f, 20, student=True), 9)
    c.expect('ticket_price(20, "weekend", True)', _call(f, 20, "weekend", True), 12)


def ex4(c):
    f = _get("add_task")
    c.expect('add_task("read")  (1st call)', _call(f, "read"), ["read"])
    c.expect('add_task("code")  (2nd call)', _call(f, "code"), ["code"])
    mine = ["plan"]
    r = _call(f, "build", mine)
    c.expect('add_task("build", ["plan"])', r, ["plan", "build"])
    c.t(mine == ["plan", "build"] and r[0] == "ok" and r[1] is mine,
        "when a list is passed in, that same list is updated and returned",
        "when the caller passes a list, append to THAT list and return it")
    ps = _sig(f)
    c.t("todo" in ps and ps["todo"].default is None, "todo defaults to None",
        "the parameter todo must default to None (not [])")
    c.expect('add_task("test", todo=[])', _call(f, "test", todo=[]), ["test"])


def ex5(c):
    f = _get("join_path")
    c.expect('join_path("home", "zj", "lab")', _call(f, "home", "zj", "lab"), "home/zj/lab")
    c.expect("join_path()", _call(f), "")
    c.expect('join_path("a", "", "b")', _call(f, "a", "", "b"), "a/b")
    c.expect('join_path("C:", "Users", sep="\\\\")', _call(f, "C:", "Users", sep="\\"), "C:\\Users")
    ps = list(_sig(f).values())
    ok = (len(ps) == 2 and ps[0].kind == P.VAR_POSITIONAL and ps[1].name == "sep"
          and ps[1].kind == P.KEYWORD_ONLY and ps[1].default == "/")
    c.t(ok, "signature is (*parts, sep=\"/\")", "signature should be exactly (*parts, sep=\"/\")")
    v = _get("course_path")
    c.t(v == "cs250/labs/week9", "course_path built by unpacking the folders list",
        "course_path is %r; it should be join_path(*folders) → 'cs250/labs/week9'" % (v,))


def ex6(c):
    f = _get("build_url")
    c.expect('build_url("https://site.com/search", q="python", page=2)',
             _call(f, "https://site.com/search", q="python", page=2), "https://site.com/search?q=python&page=2")
    c.expect('build_url("https://site.com")', _call(f, "https://site.com"), "https://site.com")
    c.expect('build_url("u", lang="en")', _call(f, "u", lang="en"), "u?lang=en")
    c.expect('build_url("u", b=1, a=2)  (keeps call order)', _call(f, "u", b=1, a=2), "u?b=1&a=2")
    c.expect('build_url("u", on=True)', _call(f, "u", on=True), "u?on=True")
    ps = list(_sig(f).values())
    c.t(len(ps) == 2 and ps[1].kind == P.VAR_KEYWORD, "signature is (base, **params)",
        "signature should be (base, **params)")
    v = _get("news_url")
    c.t(v == "https://site.com/news?lang=en&sort=new&page=1", "news_url built with **filters plus page=1",
        "news_url is %r; use build_url(\"https://site.com/news\", **filters, page=1)" % (v,))


def ex7(c):
    f = _get("plan_workout")
    c.expect('plan_workout("Zoe")', _call(f, "Zoe"),
             {"name": "Zoe", "minutes": 30, "exercises": [], "settings": {}})
    c.expect('plan_workout("Kai", 45, "squats", "plank", intensity="high", music=True)',
             _call(f, "Kai", 45, "squats", "plank", intensity="high", music=True),
             {"name": "Kai", "minutes": 45, "exercises": ["squats", "plank"],
              "settings": {"intensity": "high", "music": True}})
    c.expect('plan_workout("Mia", minutes=15)', _call(f, "Mia", minutes=15),
             {"name": "Mia", "minutes": 15, "exercises": [], "settings": {}})
    r = _call(f, "Ray", 20, "lunges")
    c.t(r[0] == "ok" and isinstance(r[1], dict) and isinstance(r[1].get("exercises"), list),
        "exercises is stored as a list", "convert the *exercises tuple to a list: list(exercises)")
    kinds = [p.kind for p in _sig(f).values()]
    c.t(kinds == [P.POSITIONAL_OR_KEYWORD, P.POSITIONAL_OR_KEYWORD, P.VAR_POSITIONAL, P.VAR_KEYWORD],
        "parameter order is required → default → *args → **kwargs",
        "order must be: name, minutes=30, *exercises, **settings")
    ps = _sig(f)
    c.t("minutes" in ps and ps["minutes"].default == 30, "minutes defaults to 30", "minutes must default to 30")


def ex8(c):
    c.mcq("q8a", "the 2nd positional value replaces the default; only the leftovers go into *tags")
    c.mcq("q8b", "which required parameter never receives a value?")
    c.mcq("q8c", "everything after * must be passed by name")
    c.mcq("q8d", "**base supplies one entry and size=\"L\" adds another; no name is repeated")


# ---------------------------------------------------------------- PART B
def ex9(c):
    f = _get("calc_discount")
    c.expect("calc_discount(50)", _call(f, 50), 40.0)
    c.expect("calc_discount(80, rate=0.25)", _call(f, 80, rate=0.25), 60.0)
    c.expect("calc_discount(19.99)", _call(f, 19.99), 15.99)
    v = _get("sale_price")
    c.t(v == 40.0, "sale_price stores the returned value", "sale_price is %r; set sale_price = calc_discount(50)" % (v,))
    c.t("discounted" not in NB and not _uses_global(f),
        "the local variable stays local (no global, no leaking)",
        "don't make `discounted` global; return the value and store it in sale_price instead "
        "(if you created a global `discounted`, run: del discounted)")


def ex10(c):
    f = _get("lose_life")
    c.expect("lose_life(3)", _call(f, 3), 2)
    c.expect("lose_life(3, damage=2)", _call(f, 3, damage=2), 1)
    c.expect("lose_life(1, 5)  (never below 0)", _call(f, 1, 5), 0)
    ps = _sig(f)
    c.t("damage" in ps and ps["damage"].default == 1, "damage defaults to 1", "damage must default to 1")
    c.t(not _uses_global(f), "no global keyword: data goes in and comes out with return",
        "this exercise must be solved WITHOUT the global keyword")
    v = _get("lives_left")
    c.t(v == 1, "lives_left is 1", "lives_left is %r; start at 5, lose 1, then lose 3 → 1" % (v,))


def ex11(c):
    f = _get("track_view")
    had = "page_views" in NB
    old = NB.get("page_views")
    NB["page_views"] = 0
    rs = [_call(f) for _ in range(3)]
    final = NB.get("page_views")
    if had:
        NB["page_views"] = old
    c.t(all(r[0] == "ok" for r in rs), "track_view() runs without errors",
        "track_view() %s" % next((_desc(r) for r in rs if r[0] == "err"), ""))
    c.t(final == 3, "3 calls changed the global page_views from 0 to 3",
        "after 3 calls the global page_views should be 3 but it is %r" % (final,))
    c.t([r[1] for r in rs] == [1, 2, 3], "track_view() returns the new count each time (1, 2, 3)",
        "track_view() should return the updated count; got %r" % ([r[1] for r in rs],))
    c.t(_uses_global(f), "uses the global keyword", "this function must use `global page_views`")


def ex12(c):
    mg = _get("make_greeter")
    g = _call(mg, "Hola")
    c.t(g[0] == "ok" and callable(g[1]), "make_greeter returns a function",
        "make_greeter(\"Hola\") should return the inner function (return greet, no parentheses)")
    if g[0] == "ok" and callable(g[1]):
        c.expect('make_greeter("Hola")("Ana")', _call(g[1], "Ana"), "Hola, Ana!")
    else:
        c.t(False, "", "can't test greet(name) until make_greeter returns a function")
    mc = _get("make_counter")
    a = _call(mc)
    if a[0] == "ok" and callable(a[1]):
        c1 = a[1]
        r = [_call(c1), _call(c1)]
        c.t([x[1] for x in r] == [1, 2], "make_counter() counts 1, 2",
            "make_counter() should give 1 then 2; got %s" % [_desc(x) for x in r])
        b = _call(mc, 10)
        c2 = b[1] if b[0] == "ok" and callable(b[1]) else None
        c.t(c2 is not None and _call(c2)[1] == 11, "make_counter(10) starts at 10 → 11",
            "make_counter(10)() should return 11")
        c.t(_call(c1)[1] == 3, "each counter keeps its own enclosed count",
            "the first counter should now be at 3. Two counters must not share a value")
        c.t("STORE_DEREF" in _ops(c1) and not _uses_global(c1),
            "inner function updates the enclosed variable with nonlocal",
            "use `nonlocal` inside the inner function (not global)")
    else:
        for _ in range(4):
            c.t(False, "", "make_counter() should return the inner function")
    c.t(not _uses_global(mg), "make_greeter only reads the enclosed variable", "don't use global here")


_BUILTINS = ["sum", "max", "min", "len", "list", "dict", "str", "int", "float", "print",
             "input", "range", "sorted", "round", "type", "id", "set", "tuple", "abs", "any", "all"]


def ex13(c):
    f = _get("score_report")
    c.expect("score_report([80, 95, 70])", _call(f, [80, 95, 70]), "total=245, best=95, avg=81.7")
    c.expect("score_report([50])", _call(f, [50]), "total=50, best=50, avg=50.0")
    bad = [n for n in getattr(getattr(f, "__code__", None), "co_varnames", ()) if n in _BUILTINS]
    c.t(not bad, "no local variable reuses a built-in name",
        "rename these local variables, they hide built-ins: %s" % bad)
    shadow = [n for n in _BUILTINS if n in NB and NB[n] is not getattr(builtins, n)]
    c.t(not shadow, "no global variable hides a built-in",
        "your notebook has globals hiding built-ins: %s. Run: del %s" % (shadow, ", ".join(shadow)))
    c.mcq("q13", "assigning to max anywhere in the function makes max local for the WHOLE function")


def ex14(c):
    add, wt = _get("add_tag"), _get("with_tag")
    tags = ["python"]
    r = _call(add, tags, "scope")
    c.t(tags == ["python", "scope"], "add_tag changes the caller's list in place",
        "after add_tag(tags, \"scope\") the caller's list should be ['python', 'scope']; it is %r" % (tags,))
    c.t(r == ("ok", None), "add_tag returns None (it mutates instead)", "add_tag should not return anything; %s" % _desc(r))
    orig = ["a"]
    r2 = _call(wt, orig, "b")
    c.expect('with_tag(["a"], "b")', r2, ["a", "b"])
    c.t(orig == ["a"], "with_tag leaves the original list unchanged",
        "with_tag changed the original list to %r; build a NEW list instead" % (orig,))
    c.t(r2[0] == "ok" and r2[1] is not orig, "with_tag returns a brand-new list object",
        "with_tag must return a new list, not the same one")
    c.mcq("q14", "for lists, nums += [3] behaves like nums.extend([3])")


def ex15(c):
    c.mcq("q15a", "color is assigned later in paint(), so it is local everywhere in paint()")
    c.mcq("q15b", "nonlocal lets inner() update outer's n, and inner() runs twice")
    c.mcq("q15c", "a global is looked up when the function RUNS, not when it is defined")
    c.mcq("q15d", "global x in b() changes the module's x; a() still returns its own local x")


# ---------------------------------------------------------------- PART C
def ex16(c):
    f = _get("shipping_cost")
    c.expect("shipping_cost(2)", _call(f, 2), 8.0)
    c.expect("shipping_cost(8)", _call(f, 8), 17.0)
    c.expect("shipping_cost(14)  (capped)", _call(f, 14), 20.0)
    c.expect("shipping_cost(4, per_kg=3)", _call(f, 4, per_kg=3), 17.0)
    c.expect("shipping_cost(30, cap=50)", _call(f, 30, cap=50), 50.0)
    ps = _sig(f)
    ok = all(k in ps for k in ("per_kg", "base", "cap")) and \
        (ps["per_kg"].default, ps["base"].default, ps["cap"].default) == (1.5, 5.0, 20.0)
    c.t(ok, "defaults are per_kg=1.5, base=5.0, cap=20.0", "defaults should be per_kg=1.5, base=5.0, cap=20.0")
    g = _get("shipping_line")
    c.expect('shipping_line("A", 2)', _call(g, "A", 2), "Order A: $8.00")
    c.expect('shipping_line("D", 4, per_kg=3)', _call(g, "D", 4, per_kg=3), "Order D: $17.00")
    c.t("shipping_cost" in getattr(getattr(g, "__code__", None), "co_names", ()),
        "shipping_line reuses shipping_cost (no copy-paste)",
        "shipping_line should CALL shipping_cost instead of repeating the formula")


EXERCISES = {
    1: ("Positional arguments", 5, 5, ex1),
    2: ("Keyword & keyword-only arguments", 6, 6, ex2),
    3: ("Default arguments", 6, 6, ex3),
    4: ("The mutable-default trap", 6, 6, ex4),
    5: ("*args", 6, 6, ex5),
    6: ("**kwargs and ** unpacking", 7, 7, ex6),
    7: ("All four kinds together", 6, 6, ex7),
    8: ("Predict: argument binding", 8, 4, ex8),
    9: ("Local scope & return", 5, 5, ex9),
    10: ("Fix UnboundLocalError without global", 6, 6, ex10),
    11: ("The global keyword (rare case)", 4, 4, ex11),
    12: ("Enclosed scope & nonlocal", 7, 7, ex12),
    13: ("Built-in scope: don't shadow", 5, 5, ex13),
    14: ("Mutating vs reassigning", 6, 6, ex14),
    15: ("Predict: the LEGB rule", 8, 4, ex15),
    16: ("Refactor a script into functions", 9, 9, ex16),
}


def _run(n):
    title, pts, ntests, fn = EXERCISES[n]
    c = _C()
    try:
        fn(c)
    except _Missing as m:
        c.lines.append("   ❌ `%s` is not defined yet. Write it, run that cell, then run this check again." % m.args[0])
    except Exception as e:
        c.lines.append("   ❌ the checker hit an error: %s: %s" % (type(e).__name__, e))
    earned = round(pts * c.ok_n / ntests, 1)
    RESULTS[n] = (earned, pts)
    return title, pts, ntests, c, earned


def check(n):
    title, pts, ntests, c, earned = _run(n)
    print("Exercise %d · %s" % (n, title))
    print("\n".join(c.lines))
    if c.ok_n == ntests:
        print("\n✅ All %d checks passed: %g/%g points. Nice work!" % (ntests, pts, pts))
    else:
        print("\n❌ %d of %d checks passed: %g/%g points. Fix the ❌ lines above and run again."
              % (c.ok_n, ntests, earned, pts))


def final_check():
    first = str(NB.get("FIRST_NAME", "")).strip()
    last = str(NB.get("LAST_NAME", "")).strip()
    total = 0.0
    print("CS250 · Lab 13 · Arguments & Scope · FINAL CHECK")
    print("=" * 60)
    for n in EXERCISES:
        title, pts, ntests, c, earned = _run(n)
        total += earned
        mark = "✅" if earned == pts else "❌"
        print("%s Ex %2d  %-38s %5g / %g" % (mark, n, title, earned, pts))
    total = round(total, 1)
    print("=" * 60)
    print("Student: %s %s" % (first or "(FIRST_NAME missing)", last or "(LAST_NAME missing)"))
    print("YOUR GRADE: %g / 100" % total)
    print()
    if not first or not last:
        print("⚠️  Fill in FIRST_NAME and LAST_NAME in the Student Info cell at the top, run it, then run this cell again.")
        return total
    fname = "%s_%s_L13.ipynb" % (first.lower().replace(" ", ""), last.lower().replace(" ", ""))
    if total == 100:
        print("🎉 100/100! Now you are ready to submit your lab.")
        print()
        print("  1. Rename this notebook to exactly:  %s" % fname)
        print("     (click the notebook title at the top left of Colab)")
        print("  2. Copy it to your GitHub: File → Save a copy in GitHub")
        print("     • Repository: your CS repo")
        print("     • File path:  lab/%s" % fname)
        print("  3. Make sure your repo is PUBLIC so your instructor can view it")
        print("     (GitHub → your repo → Settings → General → Danger Zone → Change visibility → Public).")
    else:
        print("Not done yet. Scroll up: run each ❌ exercise's check cell to see exactly what is wrong,")
        print("fix it, then run this final check again. You can submit once you reach 100.")
    return total
'''
import types as _types, __main__ as _main
grader = _types.ModuleType("l13_grader")
exec(_L13_SRC, grader.__dict__)
grader.NB = _main.__dict__
print("✅ Auto-grader loaded. Work through the exercises and run each check cell.")


✅ Auto-grader loaded. Work through the exercises and run each check cell.


## 👤 Student Info
Type your first and last name exactly as they should appear in your file name, then run the cell.

In [ ]:
FIRST_NAME = "YI"   # e.g. "Jane"
LAST_NAME = "CUI"    # e.g. "Doe"

---
# Part A · Arguments
*Parameters are **P**laceholders in the `def` line. Arguments are the **A**ctual values in the call.*

## Exercise 1 · Positional arguments (5 pts)
With positional arguments, Python matches values to parameters **by position**. Swapping them gives no error, just a wrong answer (a *logic bug*). Leaving one out raises an error.

**Tasks**
1. Write `trip_summary(city, nights, travelers)` that **returns** a string like
   `"3 traveler(s), 5 night(s) in Lima"`.
2. A teammate wrote `trip_summary(4, "Tokyo", 2)` for a trip of **2 travelers, 4 nights, in Tokyo**. Store the corrected call's result in `fixed_trip` (positional arguments only).
3. Without changing the function, what *type of error* does `trip_summary("Rome", 3)` raise? Store just its name as a string in `q1_error` (e.g. `"ValueError"`).

In [ ]:
def trip_summary(city, nights, travelers):
    # YOUR CODE HERE
    return f"{travelers} traveler(s), {nights} night(s) in {city}"
trip_summary("Lima",5,3)

fixed_trip = trip_summary("Tokyo",4,2)   # YOUR CODE HERE: call trip_summary with the arguments in the right order

q1_error = "TypeError"       # name of the error raised by trip_summary("Rome", 3)

In [ ]:
# ✔️ Check Exercise 1
grader.check(1)

Exercise 1 · Positional arguments
   ✅ trip_summary("Lima", 5, 3) → '3 traveler(s), 5 night(s) in Lima'
   ✅ trip_summary("Oslo", 1, 1) → '1 traveler(s), 1 night(s) in Oslo'
   ✅ parameters are (city, nights, travelers)
   ✅ fixed_trip has the arguments in the right order
   ✅ q1_error is correct

✅ All 5 checks passed: 5/5 points. Nice work!


## Exercise 2 · Keyword & keyword-only arguments (6 pts)
Keyword arguments (`name=value`) are matched **by name**, so order doesn't matter. Putting a bare `*` in the `def` line makes every parameter after it **keyword-only**: callers *must* name them.

**Tasks**
1. Write `create_account(username, *, email, admin=False)` that **returns** the dictionary
   `{"username": ..., "email": ..., "admin": ...}`.
   * `username` can be positional; `email` is required **and** keyword-only; `admin` is keyword-only with a default.
2. Create `my_account` by calling `create_account` with **only keyword arguments**, listed in a *different order* than the definition, with `admin=True`.

In [ ]:
def create_account(username, *, email, admin=False):
    # YOUR CODE HERE
    return {"username": username, "email": email, "admin": admin}

my_account = create_account(admin=True, email="ycui4407@student.edu",username="YI")   # YOUR CODE HERE



In [ ]:
# ✔️ Check Exercise 2
grader.check(2)

Exercise 2 · Keyword & keyword-only arguments
   ✅ create_account("ana", email="ana@sfbu.edu") → {'username': 'ana', 'email': 'ana@sfbu.edu', 'admin': False}
   ✅ create_account(email="ana@sfbu.edu", username="ana") → {'username': 'ana', 'email': 'ana@sfbu.edu', 'admin': False}
   ✅ create_account("li", email="li@sfbu.edu", admin=True) → {'username': 'li', 'email': 'li@sfbu.edu', 'admin': True}
   ✅ create_account("ana", "ana@sfbu.edu") raises TypeError (as it should)
   ✅ email is a required keyword-only parameter
   ✅ my_account was created with admin=True

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 3 · Default arguments (6 pts)
A default value in the `def` line makes a parameter optional. **Required parameters come first.** To skip one default and change a later one, use a keyword.

**Task:** write `ticket_price(age, day="weekday", student=False)` that **returns** a movie ticket price:
* Base price: **12** on a `"weekday"`, **15** on a `"weekend"`.
* Children under 12 and seniors 65+ pay **half** the base price.
* If `student` is `True`, take **3 off** the price (after the age rule).

| Call | Returns |
|---|---|
| `ticket_price(30)` | `12` |
| `ticket_price(70, day="weekend")` | `7.5` |
| `ticket_price(20, student=True)` | `9` |

In [ ]:
def ticket_price(age, day="weekday", student=False):
    # YOUR CODE HERE
    if day == "weekday":
      base_price = 12
    elif day =="weekend":
      base_price = 15
    if age < 12 or age >= 65:
      base_price *= 0.5
    if student:
      base_price -= 3
    return base_price

# Try a few calls of your own:
# print(ticket_price(30))
# print(ticket_price(20, student=True))   # skips `day` by using a keyword

In [ ]:
# ✔️ Check Exercise 3
grader.check(3)

Exercise 3 · Default arguments
   ✅ ticket_price(30) → 12
   ✅ ticket_price(30, "weekend") → 15
   ✅ ticket_price(8) → 6
   ✅ ticket_price(70, day="weekend") → 7.5
   ✅ ticket_price(20, student=True) → 9
   ✅ ticket_price(20, "weekend", True) → 12

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 4 · The mutable-default trap (6 pts)
A default value is created **once**, when `def` runs, and shared by every call that uses it. So a list default "remembers" old calls.

This to-do helper has that bug:
```python
def add_task(task, todo=[]):
    todo.append(task)
    return todo

print(add_task("read"))   # ['read']
print(add_task("code"))   # ['read', 'code']   <- "read" leaked in!
```
**Task:** fix `add_task` using the safe pattern so that every call **without** a list gets a fresh list, and a call **with** a list appends to (and returns) that list.

In [ ]:
def add_task(task, todo= None ):      # <- fix this function
    if todo is None:
      todo = []
    todo.append(task)
    return todo

print(add_task("read"))
print(add_task("code"))    # should print ['code']

my_list = ["shop"]
print(add_task("study", my_list))  # ['shop', 'study']
print(my_list)

['read']
['code']
['shop', 'study']
['shop', 'study']


In [ ]:
# ✔️ Check Exercise 4
grader.check(4)

Exercise 4 · The mutable-default trap
   ✅ add_task("read")  (1st call) → ['read']
   ✅ add_task("code")  (2nd call) → ['code']
   ✅ add_task("build", ["plan"]) → ['plan', 'build']
   ✅ when a list is passed in, that same list is updated and returned
   ✅ todo defaults to None
   ✅ add_task("test", todo=[]) → ['test']

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 5 · `*args` (6 pts)
`*parts` in a `def` packs **any number** of positional arguments into a **tuple**. In a *call*, `*some_list` does the opposite: it **unpacks** a list into separate positional arguments.

**Tasks**
1. Write `join_path(*parts, sep="/")` that **returns** the non-empty parts joined by `sep`.
   * `join_path("home", "zj", "lab")` → `"home/zj/lab"`
   * `join_path("a", "", "b")` → `"a/b"` (skip empty strings)
   * `join_path()` → `""`
   * Note: because `sep` comes **after** `*parts`, it can only be passed by keyword.
2. Given the list `folders` below, set `course_path` by calling `join_path` and **unpacking** the list with `*`.

In [ ]:
def join_path(*parts, sep="/"):
    # YOUR CODE HERE
    non_empty_parts = []
    for part in parts:
        if part != "":
            non_empty_parts.append(part)
    return sep.join(non_empty_parts)


folders = ["cs250", "labs", "week9"]
course_path = join_path(*folders)   # YOUR CODE HERE (use *folders)

In [ ]:
# ✔️ Check Exercise 5
grader.check(5)

Exercise 5 · *args
   ✅ join_path("home", "zj", "lab") → 'home/zj/lab'
   ✅ join_path() → ''
   ✅ join_path("a", "", "b") → 'a/b'
   ✅ join_path("C:", "Users", sep="\\") → 'C:\\Users'
   ✅ signature is (*parts, sep="/")
   ✅ course_path built by unpacking the folders list

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 6 · `**kwargs` and `**` unpacking (7 pts)
`**params` in a `def` collects any extra **keyword** arguments into a **dictionary** (names → keys). In a call, `**some_dict` unpacks a dictionary into keyword arguments.

**Tasks**
1. Write `build_url(base, **params)` that **returns** a web address with a query string, keeping the order the keywords were given:
   * `build_url("https://site.com/search", q="python", page=2)` → `"https://site.com/search?q=python&page=2"`
   * `build_url("https://site.com")` → `"https://site.com"` (no `?` when there are no params)
2. Using the `filters` dictionary below, set `news_url` by calling `build_url` with base `"https://site.com/news"`, **unpacking `filters`**, and adding `page=1`.

In [ ]:
def build_url(base, **params):
    # YOUR CODE HERE
    query_parts = []
    if not params:
        return base
    for key, value in params.items():
        query_parts.append(f"{key}={value}")
    return base + "?" + "&".join(query_parts)


filters = {"lang": "en", "sort": "new"}
news_url = build_url("https://site.com/news", **filters, page=1)   # YOUR CODE HERE

In [ ]:
# ✔️ Check Exercise 6
grader.check(6)

Exercise 6 · **kwargs and ** unpacking
   ✅ build_url("https://site.com/search", q="python", page=2) → 'https://site.com/search?q=python&page=2'
   ✅ build_url("https://site.com") → 'https://site.com'
   ✅ build_url("u", lang="en") → 'u?lang=en'
   ✅ build_url("u", b=1, a=2)  (keeps call order) → 'u?b=1&a=2'
   ✅ build_url("u", on=True) → 'u?on=True'
   ✅ signature is (base, **params)
   ✅ news_url built with **filters plus page=1

✅ All 7 checks passed: 7/7 points. Nice work!


## Exercise 7 · All four kinds together (6 pts)
Order inside a `def`: **required → default → `*args` → `**kwargs`**.

**Task:** write `plan_workout(name, minutes=30, *exercises, **settings)` that **returns**
```python
{"name": name, "minutes": minutes, "exercises": <list of exercises>, "settings": <dict of settings>}
```
Store `exercises` as a **list** (it arrives as a tuple).

| Call | Returns |
|---|---|
| `plan_workout("Zoe")` | `{"name": "Zoe", "minutes": 30, "exercises": [], "settings": {}}` |
| `plan_workout("Kai", 45, "squats", "plank", intensity="high")` | `{"name": "Kai", "minutes": 45, "exercises": ["squats", "plank"], "settings": {"intensity": "high"}}` |

In [ ]:
def plan_workout(name, minutes=30, *exercises, **settings):   # <- write the full parameter list
    # YOUR CODE HERE
    return {
        "name": name,
        "minutes": minutes,
        "exercises": list(exercises),
        "settings": settings
    }

In [ ]:
# ✔️ Check Exercise 7
grader.check(7)

Exercise 7 · All four kinds together
   ✅ plan_workout("Zoe") → {'name': 'Zoe', 'minutes': 30, 'exercises': [], 'settings': {}}
   ✅ plan_workout("Kai", 45, "squats", "plank", intensity="high", music=True) → {'name': 'Kai', 'minutes': 45, 'exercises': ['squats', 'plank'], 'settings': {'intensity': 'high', 'music': True}}
   ✅ plan_workout("Mia", minutes=15) → {'name': 'Mia', 'minutes': 15, 'exercises': [], 'settings': {}}
   ✅ exercises is stored as a list
   ✅ parameter order is required → default → *args → **kwargs
   ✅ minutes defaults to 30

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 8 · Predict: argument binding (8 pts)
For each snippet, **predict first**, then type the letter (`"A"`, `"B"`, `"C"` or `"D"`).

**q8a**
```python
def book(title, author="Unknown", *tags):
    print(title, author, tags)

book("Dune", "Herbert", "sci-fi")
```
A) `Dune Unknown ('Herbert', 'sci-fi')`  B) `Dune Herbert ('sci-fi',)`  C) `Dune Herbert sci-fi`  D) `TypeError`

**q8b**
```python
def mix(a, b, c=0):
    print(a, b, c)

mix(1, c=5)
```
A) `1 0 5`  B) `1 5 0`  C) `TypeError: missing 1 required positional argument: 'b'`  D) `SyntaxError`

**q8c**
```python
def send(to, *, subject, urgent=False):
    print(to, subject, urgent)

send("ana@sfbu.edu", "Hi")
```
A) `ana@sfbu.edu Hi False`  B) `TypeError: send() takes 1 positional argument but 2 were given`  C) `ana@sfbu.edu None False`  D) `SyntaxError`

**q8d**
```python
def setup(**opts):
    print(len(opts), opts)

base = {"color": "red"}
setup(**base, size="L")
```
A) `2 {'color': 'red', 'size': 'L'}`  B) `TypeError: got multiple values`  C) `1 {'color': 'red'}`  D) `SyntaxError`

In [ ]:
q8a = "B"
q8b = "C"
q8c = "B"
q8d = "A"

In [ ]:
# ✔️ Check Exercise 8
grader.check(8)

Exercise 8 · Predict: argument binding
   ✅ q8a is correct
   ✅ q8b is correct
   ✅ q8c is correct
   ✅ q8d is correct

✅ All 4 checks passed: 8/8 points. Nice work!


---
# Part B · Scope
**Scope** is where a name can be seen. Python searches **L**ocal → **E**nclosed → **G**lobal → **B**uilt-in, and the first match wins.

## Exercise 9 · Local scope & `return` (5 pts)
A variable created inside a function is **local**: it disappears when the function returns. This code crashes:
```python
def calc_discount(price):
    discounted = price * 0.8

calc_discount(50)
print(discounted)    # NameError: name 'discounted' is not defined
```
**Tasks**
1. Rewrite `calc_discount(price, rate=0.2)` so it **returns** the price after the discount, rounded to 2 decimals (`calc_discount(50)` → `40.0`).
2. Store the result of `calc_discount(50)` in a variable named `sale_price`. Do **not** create a global named `discounted`.

In [ ]:
def calc_discount(price, rate=0.2):
    # YOUR CODE HERE
    return round(price * (1 - rate), 2)

sale_price = calc_discount(50)   # YOUR CODE HERE

In [ ]:
# ✔️ Check Exercise 9
grader.check(9)

Exercise 9 · Local scope & return
   ✅ calc_discount(50) → 40.0
   ✅ calc_discount(80, rate=0.25) → 60.0
   ✅ calc_discount(19.99) → 15.99
   ✅ sale_price stores the returned value
   ✅ the local variable stays local (no global, no leaking)

✅ All 5 checks passed: 5/5 points. Nice work!


## Exercise 10 · Fix an `UnboundLocalError` (without `global`) (6 pts)
```python
lives = 3

def lose_life():
    lives -= 1       # UnboundLocalError!
```
Because `lives -= 1` *assigns* to `lives`, Python treats `lives` as local in the whole function, then tries to read it before it has a value.

**Tasks** (the preferred fix: pass in, return out)
1. Write `lose_life(lives, damage=1)` that **returns** the new number of lives, never going below `0`. Do **not** use `global`.
2. Start with `lives_left = 5`. Use `lose_life` to lose 1 life, then 3 more lives, updating `lives_left` each time.

In [ ]:
def lose_life(lives, damage=1):
    # YOUR CODE HERE
    return max(0, lives - damage)


lives_left = 5
# YOUR CODE HERE: update lives_left twice (lose 1, then lose 3)
lives_left = lose_life(lives_left)
lives_left = lose_life(lives_left, 3)

In [ ]:
# ✔️ Check Exercise 10
grader.check(10)

Exercise 10 · Fix UnboundLocalError without global
   ✅ lose_life(3) → 2
   ✅ lose_life(3, damage=2) → 1
   ✅ lose_life(1, 5)  (never below 0) → 0
   ✅ damage defaults to 1
   ✅ no global keyword: data goes in and comes out with return
   ✅ lives_left is 1

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 11 · The `global` keyword (the rare case) (4 pts)
Sometimes a program really does keep one shared, global value. `global name` tells Python: *"assign to the global one, don't make a local."* Use it sparingly.

**Task:** write `track_view()` that adds 1 to the **global** `page_views` and **returns** the new value.

In [ ]:
page_views = 0

def track_view():
    # YOUR CODE HERE
    global page_views
    page_views += 1
    return page_views

In [ ]:
# ✔️ Check Exercise 11
grader.check(11)

Exercise 11 · The global keyword (rare case)
   ✅ track_view() runs without errors
   ✅ 3 calls changed the global page_views from 0 to 3
   ✅ track_view() returns the new count each time (1, 2, 3)
   ✅ uses the global keyword

✅ All 4 checks passed: 4/4 points. Nice work!


## Exercise 12 · Enclosed scope & `nonlocal` (7 pts)
A function defined **inside** another function can *read* the outer function's variables (the **E** in LEGB). To *change* one, it needs `nonlocal`.

**Tasks**
1. Write `make_greeter(greeting)`. Inside it, define `greet(name)` that returns `f"{greeting}, {name}!"`, then **return the `greet` function** (no parentheses).
   `make_greeter("Hola")("Ana")` → `"Hola, Ana!"`
2. Write `make_counter(start=0)`. Inside it, define `step()` that adds 1 to the enclosed `count` using `nonlocal` and returns it. Return `step`.
   ```python
   c = make_counter()
   c(); c()          # 1, then 2
   make_counter(10)()   # 11
   ```

In [ ]:
def make_greeter(greeting):
    # YOUR CODE HERE
    def greet(name):
        return f"{greeting}, {name}!"
    return greet


def make_counter(start=0):
    count = start
    # YOUR CODE HERE
    def step():
        nonlocal count
        count += 1
        return count

    return step

In [ ]:
# ✔️ Check Exercise 12
grader.check(12)

Exercise 12 · Enclosed scope & nonlocal
   ✅ make_greeter returns a function
   ✅ make_greeter("Hola")("Ana") → 'Hola, Ana!'
   ✅ make_counter() counts 1, 2
   ✅ make_counter(10) starts at 10 → 11
   ✅ each counter keeps its own enclosed count
   ✅ inner function updates the enclosed variable with nonlocal
   ✅ make_greeter only reads the enclosed variable

✅ All 7 checks passed: 7/7 points. Nice work!


## Exercise 13 · Built-in scope: don't shadow built-ins (5 pts)
Built-in names (`sum`, `max`, `len`, `list` …) are searched **last**, so any variable with the same name hides them. This function crashes:
```python
def score_report(scores):
    sum = 0
    for s in scores:
        sum += s
    average = sum / len(scores)
    max = max(scores)
    return f"total={sum}, best={max}, avg={round(average, 1)}"
```
**Tasks**
1. Fix `score_report` by renaming variables so **no local variable** uses a built-in name. Keep the same return format:
   `score_report([80, 95, 70])` → `"total=245, best=95, avg=81.7"`
2. **q13:** what error does the original version raise?
   A) `TypeError: 'int' object is not callable`  B) `UnboundLocalError`  C) `NameError`  D) no error

In [ ]:
def score_report(scores):        # <- fix this function
    total_score = 0
    for s in scores:
        total_score += s
    average = total_score / len(scores)
    best_score = max(scores)
    return f"total={total_score}, best={best_score}, avg={round(average, 1)}"


print(score_report([80, 95, 70]))


q13 = "B"

total=245, best=95, avg=81.7


In [ ]:
# ✔️ Check Exercise 13
grader.check(13)

Exercise 13 · Built-in scope: don't shadow
   ✅ score_report([80, 95, 70]) → 'total=245, best=95, avg=81.7'
   ✅ score_report([50]) → 'total=50, best=50, avg=50.0'
   ✅ no local variable reuses a built-in name
   ✅ no global variable hides a built-in
   ✅ q13 is correct

✅ All 5 checks passed: 5/5 points. Nice work!


## Exercise 14 · Mutating vs reassigning (6 pts)
When you pass a list, the parameter and the caller's variable are **two names for the same list**.
* Changing what's **inside** (`.append`, `.extend`) is seen by the caller.
* **Reassigning** the parameter (`tags = ...`) only points the local name somewhere new.

**Tasks**
1. Write `add_tag(tags, tag)` that adds `tag` to the caller's list **in place** and returns nothing.
2. Write `with_tag(tags, tag)` that returns a **new** list with `tag` added and leaves the original unchanged.
3. **q14:** what does this print?
   ```python
   def grow(nums):
       nums += [3]

   data = [1, 2]
   grow(data)
   print(data)
   ```
   A) `[1, 2]`  B) `[1, 2, 3]`  C) `UnboundLocalError`  D) `TypeError`

In [ ]:
def add_tag(tags, tag):
    # YOUR CODE HERE
    tags.append(tag)


def with_tag(tags, tag):
    # YOUR CODE HERE
    new_tags = tags + [tag]
    return new_tags


q14 = "B"

In [ ]:
# ✔️ Check Exercise 14
grader.check(14)

Exercise 14 · Mutating vs reassigning
   ✅ add_tag changes the caller's list in place
   ✅ add_tag returns None (it mutates instead)
   ✅ with_tag(["a"], "b") → ['a', 'b']
   ✅ with_tag leaves the original list unchanged
   ✅ with_tag returns a brand-new list object
   ✅ q14 is correct

✅ All 6 checks passed: 6/6 points. Nice work!


## Exercise 15 · Predict: the LEGB rule (8 pts)
**Predict first**, then type the letter.

**q15a**
```python
color = "blue"

def paint():
    print(color)
    color = "red"

paint()
```
A) `blue`  B) `red`  C) `UnboundLocalError`  D) `NameError`

**q15b**
```python
def outer():
    n = 1
    def inner():
        nonlocal n
        n += 10
    inner()
    inner()
    return n

print(outer())
```
A) `21`  B) `11`  C) `1`  D) `UnboundLocalError`

**q15c**
```python
def total(items):
    return sum(items) + bonus

bonus = 5
print(total([1, 2, 3]))
```
A) `6`  B) `NameError`  C) `UnboundLocalError`  D) `11`

**q15d**
```python
x = 1

def a():
    x = 2
    def b():
        global x
        x = 3
    b()
    return x

print(a(), x)
```
A) `3 3`  B) `2 1`  C) `2 3`  D) `3 1`

In [ ]:
q15a = "C"
q15b = "A"
q15c = "D"
q15d = "C"

In [ ]:
# ✔️ Check Exercise 15
grader.check(15)

Exercise 15 · Predict: the LEGB rule
   ✅ q15a is correct
   ✅ q15b is correct
   ✅ q15c is correct
   ✅ q15d is correct

✅ All 4 checks passed: 8/8 points. Nice work!


---
# Part C · Organizing programs
## Exercise 16 · Refactor a copy-pasted script (9 pts)
This script works, but the same 4 lines are copy-pasted for every order:
```python
weight_a = 2.0
cost_a = 5.0 + weight_a * 1.5
if cost_a > 20.0:
    cost_a = 20.0
print(f"Order A: ${cost_a:.2f}")

weight_b = 8.0
cost_b = 5.0 + weight_b * 1.5
if cost_b > 20.0:
    cost_b = 20.0
print(f"Order B: ${cost_b:.2f}")

weight_c = 14.0
cost_c = 5.0 + weight_c * 1.5
if cost_c > 20.0:
    cost_c = 20.0
print(f"Order C: ${cost_c:.2f}")
```
**Tasks**
1. Write `shipping_cost(weight, per_kg=1.5, base=5.0, cap=20.0)` that **returns** `base + weight * per_kg`, but never more than `cap`, rounded to 2 decimals.
2. Write `shipping_line(order, weight, **options)` that **returns** a line like `"Order A: $8.00"`. It must **call `shipping_cost`** and pass any extra options through with `**options`, so `shipping_line("D", 4, per_kg=3)` → `"Order D: $17.00"`.
3. Print the three orders (A 2.0 kg, B 8.0 kg, C 14.0 kg) using three calls.

In [ ]:
def shipping_cost(weight, per_kg=1.5, base=5.0, cap=20.0):
    # YOUR CODE HERE
    cost = base + weight * per_kg
    return round(min(cost, cap), 2)


def shipping_line(order, weight, **options):
    # YOUR CODE HERE
    cost = shipping_cost(weight, **options)
    return f"Order {order}: ${cost:.2f}"


# YOUR CODE HERE: three calls that print orders A, B and C
print(shipping_line("A", 2.0))
print(shipping_line("B", 8.0))
print(shipping_line("C", 14.0))

Order A: $8.00
Order B: $17.00
Order C: $20.00


In [ ]:
# ✔️ Check Exercise 16
grader.check(16)

Exercise 16 · Refactor a script into functions
   ✅ shipping_cost(2) → 8.0
   ✅ shipping_cost(8) → 17.0
   ✅ shipping_cost(14)  (capped) → 20.0
   ✅ shipping_cost(4, per_kg=3) → 17.0
   ✅ shipping_cost(30, cap=50) → 50.0
   ✅ defaults are per_kg=1.5, base=5.0, cap=20.0
   ✅ shipping_line("A", 2) → 'Order A: $8.00'
   ✅ shipping_line("D", 4, per_kg=3) → 'Order D: $17.00'
   ✅ shipping_line reuses shipping_cost (no copy-paste)

✅ All 9 checks passed: 9/9 points. Nice work!


---
# 🏁 Final Check
Run this cell to see your grade. It re-tests **every** exercise from scratch, so make sure you have run all of your code cells first (Runtime → Run all works too).

When you reach **100/100** it will tell you how to submit.

In [ ]:
# 🏁 FINAL CHECK: run me last
grader.final_check();

CS250 · Lab 13 · Arguments & Scope · FINAL CHECK
✅ Ex  1  Positional arguments                       5 / 5
✅ Ex  2  Keyword & keyword-only arguments           6 / 6
✅ Ex  3  Default arguments                          6 / 6
✅ Ex  4  The mutable-default trap                   6 / 6
✅ Ex  5  *args                                      6 / 6
✅ Ex  6  **kwargs and ** unpacking                  7 / 7
✅ Ex  7  All four kinds together                    6 / 6
✅ Ex  8  Predict: argument binding                  8 / 8
✅ Ex  9  Local scope & return                       5 / 5
✅ Ex 10  Fix UnboundLocalError without global       6 / 6
✅ Ex 11  The global keyword (rare case)             4 / 4
✅ Ex 12  Enclosed scope & nonlocal                  7 / 7
✅ Ex 13  Built-in scope: don't shadow               5 / 5
✅ Ex 14  Mutating vs reassigning                    6 / 6
✅ Ex 15  Predict: the LEGB rule                     8 / 8
✅ Ex 16  Refactor a script into functions           9 / 9
Student: YI CUI
YOUR GR